# Transfer existence by pretraining condition

One table: does the trait transfer, condition by condition? Each row is a **finished set** —
every listed teacher with a student-finetune-seed-1 test endpoint on every design cell of the arm
AND a matched control on every cell; an arm short of that is dropped and named, with its coverage
counts, in the notes below the table (never a partial test).

- **Test**: arm-stratified exceedance permutation (`transfer_headline.view_permutation`, 20 000
  draws, seed 0) — each (animal, arm) block is judged against **its own** matched-control maximum
  (the untuned sender base's numbers pool distilled into the same receiver on the same cells),
  and the null relabels which pool is the control inside each cell.
- **Exceedances** = treatment runs above their block's control maximum, with their share of the
  row's runs; **Null 95th Perc. Exceedances** = the same count at the null's 95th percentile.
- Rows: one per animal that carries a finished arm in the condition, then — when at least two
  animals do — one **stratified** row (grey) whose **Animal** is `All`: the fit over every animal
  with a finished arm in that row (dog, cat, lion, pig for Pythia, elephant for DataDecide
  OLMo-1B), each finished (animal, arm) block its own stratum; a row pooling fewer than four
  animals says which in the notes.
- **# HP** = hyperparameter settings: **Teacher** = teachers per arm, **Student** = design cells
  per arm (30 = core30, 56 = core56), **Total** = the row's total runs; where the row's arms
  differ, the per-arm columns carry the distinct values (`34 / 35`).
- **Same-model** — every lineage's own teachers distilled back into itself (`160M`, `fs1–3`,
  `ds1–3`, `rerun 160M`), read for every animal on **core30**; only the diagonals that clear the
  finished-set rule enter the row, the rest are named with their counts in the notes.
- **Cross-data-order design** — the `Same init, diff. data order` condition carries the full
  12-arm set for every animal, each animal on the design its arms are finished on: dog on
  **core56**, cat / lion / pig on **core30**, where their `ds1 → ds2` arms exist.
- **Diverged students** — a student whose adapter went non-finite (`train_diverged.json` in
  its run leaf) counts as a **complete** run of its arm and is **excluded from every statistic**;
  the notes name it. An arm whose only shortfall is divergence is a full arm.
- **Selection cells** — pilot cells are never held out, here or anywhere else: a
  pilot-selected teacher set is scored on every design cell, its selection cell included, and a
  pilot cell outside the design is simply not in the design.
- **Arms** — `{S}²` is every ordered sender→receiver pair of distinct members of `S`, `\`
  subtracts the arms listed after it, a diagonal (same-model) block is written as the bare
  sender set (`{160M, fs1–3, ds1–3, rerun 160M}`), and an index range (`fs1–3`) folds consecutive
  family members.
- **Diff. model size** — the 160M ↔ 410M dog arms on core30, split by data order and
  direction: same order `160M → 410M` (small to large) and `{410M → 160M, 410M-seed2 → fs2}`
  (large to small); different order `{160M → 410M-seed1, fs1/fs2 → 410M}` and
  `{410M-seed1–3 → 160M, 410M-seed2/3 → fs1}`. `fsN` is `pythia-160m-seedN`, the full-seed-N base.
- **Pythia/OLMo Cross Arch.** — the full 3×3 seed grids, `{160M, fs1, fs3} → {Dolma1.7,
  Dolma1.7 seed2, Dolma1.7 seed3}` (core35) and its reverse (core30); 160M → Dolma1.7 is the
  complete 16-teacher arm (its top-10 re-cut is still running) and Dolma1.7 → 160M the
  seed-robust-8 arm, so every OLMo sender carries the same 8 teachers.
- **DataDecide OLMo-1B same-model** — the three recipes' own canonical teachers distilled
  back into themselves at teacher seed 0 (`{Dolma1.7, DCLM, C4}`, core35).
- A bold green **P-value** is < 0.05.

Logic lives in `analysis/transfer_conditions.py` (a re-cut of §0.0 of
`notebooks/transfer_significance.ipynb`); the LaTeX rendering of this same table is written to
`notebooks/latex/transfer_conditions_table.tex`. Cosmetic LaTeX changes (captions, headers, column widths) need no notebook run: edit the renderer and run `uv run python -m scripts.render_transfer_condition_tables`, which re-renders all three tables from the pickled frames; a DATA refresh needs this notebook.

In [1]:
import json
from subliminal_learning.analysis import transfer_conditions as tc
from subliminal_learning.analysis import transfer_headline as th
from subliminal_learning.analysis.setup import bootstrap

REPO, OUTPUT_ROOT = bootstrap()     # repo-anchored: every path below is absolute
ANIMALS = ["dog", "cat", "lion", "pig"]
TEACHERS = {a: json.loads((REPO / f"run_logs/registry/{a}_top10_kband_configs.json").read_text())["param_tags"] for a in ANIMALS}
CONDITIONS = tc.condition_blocks(REPO, TEACHERS, diverged=th.DIVERGED_EXCLUDE)   # a diverged student is a COMPLETE run, excluded from every statistic
TABLE = tc.condition_table(CONDITIONS, n_perm=20000)
tc.style_condition_table(TABLE)

In [2]:
from IPython.display import Markdown, display

display(Markdown("\n".join(f"- {n}" for n in tc.condition_notes(CONDITIONS))))

- Pythia, Same-model — design core30; dropped ds1→ds1 [cat]: not started; dropped ds2→ds2 [cat]: not started; dropped ds3→ds3 [cat]: not started; dropped pythia-160m-pilereprov1-seed1234→pythia-160m-pilereprov1-seed1234 [cat]: not started; dropped ds1→ds1 [lion]: not started; dropped ds2→ds2 [lion]: not started; dropped ds3→ds3 [lion]: not started; dropped pythia-160m-pilereprov1-seed1234→pythia-160m-pilereprov1-seed1234 [lion]: not started; dropped ds1→ds1 [pig]: not started; dropped ds2→ds2 [pig]: not started; dropped ds3→ds3 [pig]: not started; dropped pythia-160m-pilereprov1-seed1234→pythia-160m-pilereprov1-seed1234 [pig]: not started
- Same init, diff. data order — design core56 (dog) / core30 (cat, lion, pig); cat / lion / pig are read on core30: their ds1 → ds2 arms exist only there; dropped ds2→orig [cat]: not started; dropped ds2→ds1 [cat]: not started; dropped ds2→ds3 [cat]: not started; dropped ds3→orig [cat]: not started; dropped ds3→ds1 [cat]: not started; dropped ds3→ds2 [cat]: not started; dropped ds2→orig [lion]: not started; dropped ds2→ds1 [lion]: not started; dropped ds2→ds3 [lion]: not started; dropped ds3→orig [lion]: not started; dropped ds3→ds1 [lion]: not started; dropped ds3→ds2 [lion]: not started; dropped ds2→orig [pig]: not started; dropped ds2→ds1 [pig]: not started; dropped ds2→ds3 [pig]: not started; dropped ds3→orig [pig]: not started; dropped ds3→ds1 [pig]: not started; dropped ds3→ds2 [pig]: not started
- Diff. init, same data order — design core30
- Diff. init, diff. data order — design core56; 1 diverged run excluded (teacher ep20_lr1e-4_r8a16, cell ep20_lr2e-4_r64a8)
- Diff. model size, same data order, small to large — design core30; strays excluded: ep10_lr1e-4_r16a64, ep10_lr1e-4_r32a32, ep10_lr2e-4_r16a16, ep15_lr1e-4_r32a32, ep15_lr1e-4_r8a16, ep15_lr2e-4_r8a8, ep1_lr1e-3_r32a64, ep1_lr1e-3_r64a64, ep20_lr1e-4_r32a32, ep20_lr1e-4_r8a32, ep20_lr2e-4_r8a8, ep2_lr1e-3_r8a16, ep2_lr5e-4_r16a64, ep2_lr5e-4_r64a64, ep3_lr2e-4_r64a64, ep3_lr5e-4_r16a32, ep3_lr5e-4_r32a32, ep3_lr5e-4_r8a32, ep5_lr2e-4_r16a32, ep5_lr2e-4_r16a64, ep5_lr2e-4_r32a64, ep5_lr5e-4_r16a16, ep7_lr1e-4_r64a64, ep7_lr2e-4_r8a32; strays excluded: ep5_lr5e-4_r8a16
- Diff. model size, same data order, large to small — design core30; strays excluded: ep10_lr1e-4_r128a32, ep10_lr1e-4_r16a64, ep10_lr1e-4_r32a32, ep10_lr1e-4_r32a64, ep10_lr1e-4_r64a64, ep10_lr2e-4_r32a8, ep10_lr2e-4_r64a16, ep10_lr5e-5_r32a128, ep10_lr5e-5_r32a64, ep10_lr5e-5_r64a64, ep1_lr2e-4_r32a256, ep1_lr5e-4_r32a128, ep20_lr1e-4_r128a32, ep20_lr1e-4_r32a32, ep20_lr1e-4_r64a32, ep20_lr2e-4_r16a8, ep20_lr2e-4_r64a8, ep20_lr2e-5_r32a256, ep20_lr5e-5_r32a64, ep2_lr1e-3_r32a16, ep5_lr1e-3_r32a4, ep5_lr1e-4_r32a128, ep5_lr1e-4_r32a64, ep5_lr2e-4_r16a32, ep5_lr2e-4_r8a32, ep5_lr5e-5_r128a256, ep7_lr1e-4_r32a64, ep7_lr2e-4_r32a32
- Diff. model size, diff. data order, small to large — design core30; strays excluded: ep10_lr1e-4_r16a64, ep10_lr1e-4_r32a32, ep10_lr2e-4_r16a16, ep15_lr1e-4_r32a32, ep15_lr1e-4_r8a16, ep15_lr2e-4_r8a8, ep1_lr1e-3_r32a64, ep1_lr1e-3_r64a64, ep20_lr1e-4_r32a32, ep20_lr1e-4_r8a32, ep20_lr2e-4_r8a8, ep2_lr1e-3_r8a16, ep2_lr5e-4_r16a64, ep2_lr5e-4_r64a64, ep3_lr2e-4_r64a64, ep3_lr5e-4_r16a32, ep3_lr5e-4_r32a32, ep3_lr5e-4_r8a32, ep5_lr2e-4_r16a32, ep5_lr2e-4_r16a64, ep5_lr2e-4_r32a64, ep5_lr5e-4_r16a16, ep7_lr1e-4_r64a64, ep7_lr2e-4_r8a32; strays excluded: ep10_lr1e-4_r32a64, ep5_lr2e-4_r64a64, ep5_lr5e-4_r8a16, ep7_lr1e-4_r32a64, ep7_lr2e-4_r32a32; strays excluded: ep5_lr5e-4_r8a16
- Diff. model size, diff. data order, large to small — design core30; strays excluded: ep10_lr1e-4_r16a64, ep10_lr1e-4_r32a64, ep10_lr1e-4_r64a32, ep10_lr1e-4_r64a64, ep10_lr2e-4_r128a16, ep10_lr2e-4_r16a16, ep10_lr2e-4_r8a8, ep10_lr5e-4_r8a4, ep10_lr5e-5_r32a128, ep1_lr1e-3_r128a64, ep1_lr1e-3_r16a32, ep1_lr1e-3_r8a32, ep1_lr2e-3_r16a32, ep20_lr1e-4_r16a32, ep20_lr2e-4_r64a8, ep20_lr2e-4_r8a8, ep20_lr2e-5_r128a256, ep20_lr5e-5_r16a64, ep2_lr1e-3_r8a16, ep5_lr1e-3_r16a4, ep5_lr1e-3_r4a4, ep5_lr1e-4_r16a128, ep5_lr1e-4_r16a64, ep5_lr5e-4_r64a8
- Diff. model arch., diff. data corpus, small to large — design core35; the 3×3 Pythia-160M seed × Dolma1.7 seed grid; 160M → Dolma1.7 is the complete 16-teacher arm (its top-10 re-cut is still running); strays excluded: ep10_lr1e-4_r16a64, ep10_lr1e-4_r32a64, ep15_lr1e-4_r8a16, ep1_lr1e-3_r32a64, ep1_lr1e-3_r64a64, ep20_lr1e-4_r32a32, ep20_lr2e-4_r8a8, ep2_lr5e-4_r64a64, ep3_lr5e-4_r16a32, ep3_lr5e-4_r32a32, ep3_lr5e-4_r8a32, ep5_lr2e-4_r16a32, ep5_lr2e-4_r32a64, ep5_lr2e-4_r64a64, ep5_lr5e-4_r16a16, ep7_lr1e-4_r32a64, ep7_lr1e-4_r64a64, ep7_lr2e-4_r32a32
- Diff. model arch., diff. data corpus, large to small — design core30; the 3×3 Dolma1.7 seed × Pythia-160M seed grid; Dolma1.7 → 160M is the seed-robust-8 arm (not the 11-teacher one), so every sender carries the same 8 seed-robust teachers; strays excluded: ep10_lr2e-3_r8a4, ep10_lr2e-4_r128a256, ep5_lr1e-3_r16a64, ep5_lr1e-3_r2a64, ep5_lr1e-3_r4a64, ep5_lr1e-3_r64a512, ep5_lr2e-3_r16a128, ep5_lr2e-3_r256a128, ep5_lr2e-3_r32a16, ep5_lr2e-3_r64a128, ep5_lr2e-4_r256a512, ep5_lr5e-3_r128a4, ep5_lr5e-3_r64a4, ep5_lr5e-4_r128a256, ep5_lr5e-4_r16a128, ep5_lr5e-4_r256a256, ep5_lr5e-4_r32a256
- DataDecide OLMo-1B, Same-model — design core35; each recipe's own canonical teachers distilled back into itself (teacher seed 0); strays excluded: ep10_lr1e-3_r16a16, ep10_lr1e-3_r256a256, ep10_lr1e-3_r8a8, ep10_lr1e-4_r64a64, ep10_lr2e-3_r32a64, ep10_lr5e-3_r16a8, ep10_lr5e-3_r32a8, ep10_lr5e-3_r4a8, ep10_lr5e-3_r8a8, ep10_lr5e-4_r16a16, ep10_lr5e-4_r16a32, ep10_lr5e-4_r32a32, ep10_lr5e-4_r32a64, ep10_lr5e-4_r8a8, ep20_lr1e-3_r16a16, ep20_lr1e-4_r16a32, ep20_lr1e-4_r8a16, ep20_lr5e-4_r16a16, ep20_lr5e-4_r16a32, ep20_lr5e-4_r32a32, ep20_lr5e-4_r32a64, ep20_lr5e-4_r64a64, ep20_lr5e-4_r8a8, ep2_lr5e-4_r32a64, ep5_lr1e-3_r128a128, ep5_lr1e-3_r128a512, ep5_lr1e-3_r32a64, ep5_lr1e-3_r64a128, ep5_lr1e-3_r64a64, ep5_lr2e-3_r16a32, ep5_lr2e-3_r16a64, ep5_lr2e-3_r32a32, ep5_lr2e-3_r64a64, ep5_lr2e-3_r8a8, ep5_lr5e-3_r16a16, ep5_lr5e-3_r256a16, ep5_lr5e-3_r64a16, ep5_lr5e-3_r8a8
- Same init, diff. data corpus — design core35; the DataDecide recipes share the seed-default init byte-for-byte; strays excluded: ep10_lr1e-3_r128a32, ep10_lr1e-3_r256a32, ep10_lr1e-3_r64a32, ep10_lr2e-3_r32a4, ep10_lr2e-3_r8a4, ep10_lr2e-4_r128a256, ep10_lr5e-4_r16a64, ep10_lr5e-4_r8a64, ep5_lr1e-3_r16a64, ep5_lr1e-3_r256a64, ep5_lr1e-3_r2a64, ep5_lr1e-3_r32a64, ep5_lr1e-3_r8a64, ep5_lr2e-3_r32a16, ep5_lr2e-4_r256a512, ep5_lr5e-3_r8a4, ep5_lr5e-4_r16a256, ep5_lr5e-4_r256a256; strays excluded: ep10_lr1e-3_r256a256, ep10_lr1e-4_r64a64, ep10_lr2e-3_r32a64, ep10_lr5e-3_r16a8, ep10_lr5e-3_r32a8, ep10_lr5e-3_r4a8, ep10_lr5e-3_r8a8, ep10_lr5e-4_r32a64, ep20_lr1e-4_r16a32, ep20_lr1e-4_r8a16, ep2_lr5e-4_r32a64, ep5_lr1e-3_r128a128, ep5_lr1e-3_r128a512, ep5_lr1e-3_r64a128, ep5_lr1e-3_r64a64, ep5_lr2e-3_r16a64, ep5_lr2e-3_r64a64, ep5_lr5e-3_r16a16, ep5_lr5e-3_r256a16, ep5_lr5e-3_r64a16, ep5_lr5e-3_r8a8; strays excluded: ep10_lr1e-3_r256a256, ep10_lr5e-3_r16a8, ep10_lr5e-3_r32a8, ep10_lr5e-3_r4a8, ep10_lr5e-4_r32a64, ep5_lr1e-3_r128a128, ep5_lr1e-3_r128a512, ep5_lr1e-3_r64a128, ep5_lr1e-3_r64a64, ep5_lr2e-3_r16a64, ep5_lr2e-3_r64a64, ep5_lr5e-3_r16a16, ep5_lr5e-3_r256a16, ep5_lr5e-3_r64a16, ep5_lr5e-3_r8a8; strays excluded: ep5_lr1e-3_r256a64, ep5_lr1e-3_r4a64, ep5_lr1e-3_r8a64, ep5_lr2e-3_r128a16, ep5_lr2e-3_r16a16, ep5_lr2e-3_r32a16, ep5_lr2e-3_r4a16, ep5_lr5e-3_r128a16, ep5_lr5e-3_r16a4, ep5_lr5e-3_r32a4, ep5_lr5e-3_r4a8, ep5_lr5e-3_r64a4, ep5_lr5e-3_r8a4, ep5_lr5e-4_r128a256, ep5_lr5e-4_r16a256, ep5_lr5e-4_r256a512, ep5_lr5e-4_r64a256; strays excluded: ep5_lr1e-3_r256a64, ep5_lr1e-3_r4a64, ep5_lr1e-3_r8a64, ep5_lr2e-3_r128a16, ep5_lr2e-3_r16a16, ep5_lr5e-3_r128a16, ep5_lr5e-3_r16a4, ep5_lr5e-3_r4a4, ep5_lr5e-3_r4a8, ep5_lr5e-3_r64a4, ep5_lr5e-3_r8a4, ep5_lr5e-4_r128a256, ep5_lr5e-4_r16a256, ep5_lr5e-4_r256a512, ep5_lr5e-4_r64a256
- Diff. init, diff. data corpus — design core35
- Diff. init, same data corpus — design core35; the reverse Dolma1.7 seed2 → Dolma1.7 direction exists only as a 2-teacher core30 arm and is left out until it is rerun at full scale
- {S} → seed k: each corpus of S to its own seed-k model; {S}² → seed k: every cross-corpus pair of S, receivers at seed k; {S} → {R} seed k: every sender of S into every receiver of R at seed k; X → seed k: corpus X to its own seed-k model

In [3]:
TEX = tc.condition_table_latex(TABLE)
(REPO / "notebooks/latex/transfer_conditions_table.tex").write_text(TEX)
TABLE.to_pickle(REPO / "notebooks/latex/transfer_conditions_table.pkl")   # the render-only path
print(TEX)

% requires \usepackage{booktabs,graphicx}
\begin{table}[!hp]
\centering
\footnotesize
\setlength{\tabcolsep}{3pt}
\resizebox{\ifdim\width>\textwidth\textwidth\else\width\fi}{!}{%
\begin{tabular}{@{}l l l r r r r l r r r@{}}
\toprule
 &  &  &  & \multicolumn{3}{c}{\# HP} &  &  &  &  \\
\cmidrule(lr){5-7}
\begin{tabular}[c]{@{}l@{}}Model \\ Family\end{tabular} & Pretraining Condition & Arms & \# Arms & Teacher & Student & Total & Animal & Exceedances & \begin{tabular}[c]{@{}r@{}}Null 95th Perc. \\ Exceedances\end{tabular} & P-value \\
\midrule
Pythia & Same-model & \{160M, fs1--3, ds1--3, & 8 & 10 & 30 & 2400 & dog & 2303 (96.0\%) & 120 (5.0\%) & \textbf{5e-5} \\
 &  & rerun 160M\} &  &  &  &  &  &  &  &  \\
 &  & \{160M, fs1--3\} & 4 & 10 & 30 & 1200 & cat & 1159 (96.6\%) & 71 (5.9\%) & \textbf{5e-5} \\
 &  &  &  &  &  &  & lion & 1178 (98.2\%) & 72 (6.0\%) & \textbf{5e-5} \\
 &  &  &  &  &  &  & pig & 1166 (97.2\%) & 72 (6.0\%) & \textbf{5e-5} \\
 &  & \{160M, fs1--3, ds1--3, & 20 & 10

## Which condition transfers more? All pairings

Second table, same four Pythia conditions the first table's `160M / ds1-3 / fs1-3 / wc1-2 /
rerun 160M` lineages support (Same-model, Same init diff. data order, Diff. init same
data order, Diff. init diff. data order): does condition **A**'s SET of arms transfer
more than **B**'s? The unit relabelled under the null is the ARM (the model pair), the stratum is
(animal, student cell), and the statistics are read on the runs inside their stratum
(`transfer_significance_io.compare_families`, the same machinery as §3 of
`notebooks/transfer_significance.ipynb`).

- **A before B** — every pairing is oriented by the ORDER of the conditions in the table above
  (A is the earlier one), fixed before any number is read; which side is stronger is the **SIGN**
  of the contrast, never the orientation.
- **Diff. in Means** — A minus B of the per-stratum mean gap.
- **Win Prob.** — the per-stratum probability an A run beats a B run (stratified Mann-Whitney,
  ties 1/2).
- Both P-values are **TWO-SIDED** (tail doubling,
  `min(1, 2 · min(P(null ≥ obs), P(null ≤ obs)))`): which condition is the stronger one was not
  fixed in advance. Bold green = p < 0.05.
- **Treatment Effect Mode** — `Δ (p_student − p_control)` (written out on the first row, then
  `Δ`) = treatment − cell control; `Δ / prior` = that gap over the receiver's base prior
  (PRIMARY; receivers differ in prior across conditions). The two modes are adjacent rows of the
  same animal.
- **Designs** — all six pairings on **core30**, the only design every condition covers (the
  same-model controls and the `wc` arms exist there); `Same init, diff. data order` vs
  `Diff. init, diff. data order` is repeated on **core56**, which both of those cover.
  The design is not a column: it is the **# HP · Student** cell (`30` = core30, `56` = core56);
  **# HP · Teacher** is the teachers per arm and **# HP · Total A vs B** each side's run count,
  `1200 vs 3600`. Where the two sides differ, the per-arm columns carry the distinct values.
- **Arms** — same notation as the table above: `{S}²`, `\`, bare sender sets for diagonals,
  `fs1–3` (legend there).
- The `Diff. init, same data order` arms are `wc1 ↔ wc2` plus `{wc1, wc2} → rerun 160M`
  (`prv1`), exactly the arms finished in the existence table; the modern-tree `pr*` / `dclm*`
  arms are not in this table.
- **Cross-data-order design** — that condition's blocks are read where the existence table
  reads them: dog on core56, cat / lion / pig on core30 (their `ds1 → ds2` arms exist only
  there, so that arm enters the core30 pairings and not the core56 repeat).
- **Same blocks as the table above** — the comparison reads EXACTLY the (animal, arm) blocks
  the existence table counts as finished (`tc.unfinished_blocks(...)` → `exclude_blocks`); a
  block that clears `compare_families`' full-design rule but not the existence rule (every
  listed teacher on every cell) is listed under the table as *excluded: unfinished in the
  existence table*.
- **DataDecide OLMo-1B** — the four OLMo conditions (`Same-model`, `Same init, diff. data
  corpus`, `Diff. init, diff. data corpus`, `Diff. init, same data corpus`) are compared
  amongst themselves in a second family block: **elephant only** (so no `All` row — its
  stratified fit would repeat the animal's own row) on **core35**, with exactly the finished
  arms the existence table shows (3 / 6 / 4 / 3); the same-model arms are the three recipes'
  own canonical teachers distilled back into themselves at teacher seed 0. Their registry arms carry per-arm pilot-selected teacher sets, which
  `compare_families`' base-token loader cannot express, so the comparison is built from the
  **already-loaded existence matrices** (`tc.compare_condition_matrices`) and then handed to the
  same permutation tests; on a Pythia pairing the two paths agree exactly.
- Rows: one per animal carrying a usable comparison, then the grey stratified row, whose
  **Animal** is `All` (every animal pooled in that comparison). Blocks the coverage rule kept out are listed under the table.

The LaTeX rendering is written to `notebooks/latex/transfer_condition_comparisons_table.tex` (and the frame pickled beside it, for `scripts/render_transfer_condition_tables.py`).


In [4]:
from subliminal_learning.analysis import transfer_significance_io as tio

CORE30, CORE56 = REPO / "run_logs/registry/core30_cells.json", REPO / "run_logs/registry/core56_cells.json"
ARMS = tc.comparison_arms()
CDO, XSEED = tc.COMPARISON_CONDITIONS[1], tc.COMPARISON_CONDITIONS[3]
SPECS = [(a, b, "core30", CORE30) for a, b in tc.comparison_pairings()] + [(CDO, XSEED, "core56", CORE56)]
EXCLUDE = {d: tc.unfinished_blocks(CONDITIONS, ARMS, ANIMALS, design=d) for d in ("core30", "core56")}   # exactly the existence table's finished (animal, arm) blocks, per design
RESULTS = {(a, b, d): tio.compare_families(ARMS[a], ARMS[b], ANIMALS, TEACHERS, cells=cells, n_perm=20000, exclude_blocks=EXCLUDE[d]) for a, b, d, cells in SPECS}
PYTHIA_TABLE, COMPARISON_NOTES = tc.comparison_table(RESULTS, ARMS)

OLMO_ARMS = tc.olmo_comparison_arms(CONDITIONS)     # exactly the existence table's finished arms
OLMO_FINISHED = tc.finished_arms_by_condition(CONDITIONS)     # one family: ``Same-model`` names a Pythia block too
OLMO_RESULTS = {(a, b, "core35"): tc.compare_condition_matrices(
    OLMO_FINISHED[a], OLMO_FINISHED[b], family_a=a, family_b=b, n_perm=20000)
    for a, b in tc.comparison_pairings(tc.OLMO_COMPARISON_CONDITIONS)}
OLMO_TABLE, _ = tc.comparison_table(OLMO_RESULTS, OLMO_ARMS, family=tc.OLMO_FAMILY, names=tc.OLMO_NAMES)
COMPARISONS = tc.merge_comparison_tables([PYTHIA_TABLE, OLMO_TABLE])
if COMPARISON_NOTES:
    display(Markdown("**Blocks kept out by the coverage rule**\n" + "\n".join(
        f"- **{a} vs {b} · {d}** — {t}" for (a, b, d), t in COMPARISON_NOTES.items())))
display(Markdown("\n".join(f"- {n}" for n in tc.seed_factor_notes(
    [*COMPARISONS["Arms A"], *COMPARISONS["Arms B"]]))))
tc.style_comparison_table(COMPARISONS)

**Blocks kept out by the coverage rule**
- **Same-model vs Same init, diff. data order · core30** — cat ds1->ds1: excluded: unfinished in the existence table; cat ds2->ds2: excluded: unfinished in the existence table; cat ds3->ds3: excluded: unfinished in the existence table; cat pythia-160m-pilereprov1-seed1234->pythia-160m-pilereprov1-seed1234: excluded: unfinished in the existence table; cat ds2->orig: excluded: unfinished in the existence table; cat ds2->ds1: excluded: unfinished in the existence table; cat ds2->ds3: excluded: unfinished in the existence table; cat ds3->orig: excluded: unfinished in the existence table; cat ds3->ds1: excluded: unfinished in the existence table; cat ds3->ds2: excluded: unfinished in the existence table; lion ds1->ds1: excluded: unfinished in the existence table; lion ds2->ds2: excluded: unfinished in the existence table; lion ds3->ds3: excluded: unfinished in the existence table; lion pythia-160m-pilereprov1-seed1234->pythia-160m-pilereprov1-seed1234: excluded: unfinished in the existence table; lion ds2->orig: excluded: unfinished in the existence table; lion ds2->ds1: excluded: unfinished in the existence table; lion ds2->ds3: excluded: unfinished in the existence table; lion ds3->orig: excluded: unfinished in the existence table; lion ds3->ds1: excluded: unfinished in the existence table; lion ds3->ds2: excluded: unfinished in the existence table; pig ds1->ds1: excluded: unfinished in the existence table; pig ds2->ds2: excluded: unfinished in the existence table; pig ds3->ds3: excluded: unfinished in the existence table; pig pythia-160m-pilereprov1-seed1234->pythia-160m-pilereprov1-seed1234: excluded: unfinished in the existence table; pig ds2->orig: excluded: unfinished in the existence table; pig ds2->ds1: excluded: unfinished in the existence table; pig ds2->ds3: excluded: unfinished in the existence table; pig ds3->orig: excluded: unfinished in the existence table; pig ds3->ds1: excluded: unfinished in the existence table; pig ds3->ds2: excluded: unfinished in the existence table
- **Same-model vs Diff. init, same data order · core30** — cat ds1->ds1: excluded: unfinished in the existence table; cat ds2->ds2: excluded: unfinished in the existence table; cat ds3->ds3: excluded: unfinished in the existence table; cat pythia-160m-pilereprov1-seed1234->pythia-160m-pilereprov1-seed1234: excluded: unfinished in the existence table; lion ds1->ds1: excluded: unfinished in the existence table; lion ds2->ds2: excluded: unfinished in the existence table; lion ds3->ds3: excluded: unfinished in the existence table; lion pythia-160m-pilereprov1-seed1234->pythia-160m-pilereprov1-seed1234: excluded: unfinished in the existence table; pig ds1->ds1: excluded: unfinished in the existence table; pig ds2->ds2: excluded: unfinished in the existence table; pig ds3->ds3: excluded: unfinished in the existence table; pig pythia-160m-pilereprov1-seed1234->pythia-160m-pilereprov1-seed1234: excluded: unfinished in the existence table; pig pythia-160m-weightc-seed1->pythia-160m-pilereprov1-seed1234: excluded: unfinished in the existence table; pig pythia-160m-weightc-seed2->pythia-160m-pilereprov1-seed1234: excluded: unfinished in the existence table; pig pythia-160m-pilereprov1-seed1234->pythia-160m-weightc-seed1: excluded: unfinished in the existence table; pig pythia-160m-pilereprov1-seed1234->pythia-160m-weightc-seed2: excluded: unfinished in the existence table
- **Same-model vs Diff. init, diff. data order · core30** — cat ds1->ds1: excluded: unfinished in the existence table; cat ds2->ds2: excluded: unfinished in the existence table; cat ds3->ds3: excluded: unfinished in the existence table; cat pythia-160m-pilereprov1-seed1234->pythia-160m-pilereprov1-seed1234: excluded: unfinished in the existence table; lion ds1->ds1: excluded: unfinished in the existence table; lion ds2->ds2: excluded: unfinished in the existence table; lion ds3->ds3: excluded: unfinished in the existence table; lion pythia-160m-pilereprov1-seed1234->pythia-160m-pilereprov1-seed1234: excluded: unfinished in the existence table; pig ds1->ds1: excluded: unfinished in the existence table; pig ds2->ds2: excluded: unfinished in the existence table; pig ds3->ds3: excluded: unfinished in the existence table; pig pythia-160m-pilereprov1-seed1234->pythia-160m-pilereprov1-seed1234: excluded: unfinished in the existence table
- **Same init, diff. data order vs Diff. init, same data order · core30** — cat ds2->orig: excluded: unfinished in the existence table; cat ds2->ds1: excluded: unfinished in the existence table; cat ds2->ds3: excluded: unfinished in the existence table; cat ds3->orig: excluded: unfinished in the existence table; cat ds3->ds1: excluded: unfinished in the existence table; cat ds3->ds2: excluded: unfinished in the existence table; lion ds2->orig: excluded: unfinished in the existence table; lion ds2->ds1: excluded: unfinished in the existence table; lion ds2->ds3: excluded: unfinished in the existence table; lion ds3->orig: excluded: unfinished in the existence table; lion ds3->ds1: excluded: unfinished in the existence table; lion ds3->ds2: excluded: unfinished in the existence table; pig ds2->orig: excluded: unfinished in the existence table; pig ds2->ds1: excluded: unfinished in the existence table; pig ds2->ds3: excluded: unfinished in the existence table; pig ds3->orig: excluded: unfinished in the existence table; pig ds3->ds1: excluded: unfinished in the existence table; pig ds3->ds2: excluded: unfinished in the existence table; pig pythia-160m-weightc-seed1->pythia-160m-pilereprov1-seed1234: excluded: unfinished in the existence table; pig pythia-160m-weightc-seed2->pythia-160m-pilereprov1-seed1234: excluded: unfinished in the existence table; pig pythia-160m-pilereprov1-seed1234->pythia-160m-weightc-seed1: excluded: unfinished in the existence table; pig pythia-160m-pilereprov1-seed1234->pythia-160m-weightc-seed2: excluded: unfinished in the existence table
- **Same init, diff. data order vs Diff. init, diff. data order · core30** — cat ds2->orig: excluded: unfinished in the existence table; cat ds2->ds1: excluded: unfinished in the existence table; cat ds2->ds3: excluded: unfinished in the existence table; cat ds3->orig: excluded: unfinished in the existence table; cat ds3->ds1: excluded: unfinished in the existence table; cat ds3->ds2: excluded: unfinished in the existence table; lion ds2->orig: excluded: unfinished in the existence table; lion ds2->ds1: excluded: unfinished in the existence table; lion ds2->ds3: excluded: unfinished in the existence table; lion ds3->orig: excluded: unfinished in the existence table; lion ds3->ds1: excluded: unfinished in the existence table; lion ds3->ds2: excluded: unfinished in the existence table; pig ds2->orig: excluded: unfinished in the existence table; pig ds2->ds1: excluded: unfinished in the existence table; pig ds2->ds3: excluded: unfinished in the existence table; pig ds3->orig: excluded: unfinished in the existence table; pig ds3->ds1: excluded: unfinished in the existence table; pig ds3->ds2: excluded: unfinished in the existence table
- **Diff. init, same data order vs Diff. init, diff. data order · core30** — pig pythia-160m-weightc-seed1->pythia-160m-pilereprov1-seed1234: excluded: unfinished in the existence table; pig pythia-160m-weightc-seed2->pythia-160m-pilereprov1-seed1234: excluded: unfinished in the existence table; pig pythia-160m-pilereprov1-seed1234->pythia-160m-weightc-seed1: excluded: unfinished in the existence table; pig pythia-160m-pilereprov1-seed1234->pythia-160m-weightc-seed2: excluded: unfinished in the existence table
- **Same init, diff. data order vs Diff. init, diff. data order · core56** — cat ds1->orig: excluded: unfinished in the existence table; cat ds1->ds2: excluded: unfinished in the existence table; cat ds1->ds3: excluded: unfinished in the existence table; cat ds2->orig: excluded: unfinished in the existence table; cat ds2->ds1: excluded: unfinished in the existence table; cat ds2->ds3: excluded: unfinished in the existence table; cat ds3->orig: excluded: unfinished in the existence table; cat ds3->ds1: excluded: unfinished in the existence table; cat ds3->ds2: excluded: unfinished in the existence table; lion ds1->orig: excluded: unfinished in the existence table; lion ds1->ds2: excluded: unfinished in the existence table; lion ds1->ds3: excluded: unfinished in the existence table; lion ds2->orig: excluded: unfinished in the existence table; lion ds2->ds1: excluded: unfinished in the existence table; lion ds2->ds3: excluded: unfinished in the existence table; lion ds3->orig: excluded: unfinished in the existence table; lion ds3->ds1: excluded: unfinished in the existence table; lion ds3->ds2: excluded: unfinished in the existence table; pig ds1->orig: excluded: unfinished in the existence table; pig ds1->ds2: excluded: unfinished in the existence table; pig ds1->ds3: excluded: unfinished in the existence table; pig ds2->orig: excluded: unfinished in the existence table; pig ds2->ds1: excluded: unfinished in the existence table; pig ds2->ds3: excluded: unfinished in the existence table; pig ds3->orig: excluded: unfinished in the existence table; pig ds3->ds1: excluded: unfinished in the existence table; pig ds3->ds2: excluded: unfinished in the existence table

- {S} → seed k: each corpus of S to its own seed-k model; {S}² → seed k: every cross-corpus pair of S, receivers at seed k; {S} → {R} seed k: every sender of S into every receiver of R at seed k; X → seed k: corpus X to its own seed-k model

In [5]:
COMPARISON_TEX = tc.comparison_table_latex(COMPARISONS)
(REPO / "notebooks/latex/transfer_condition_comparisons_table.tex").write_text(COMPARISON_TEX)
COMPARISONS.to_pickle(REPO / "notebooks/latex/transfer_condition_comparisons_table.pkl")   # the render-only path
print(COMPARISON_TEX)

% requires \usepackage{booktabs,graphicx}
\begin{table}[!hp]
\centering
\scriptsize
\setlength{\tabcolsep}{3pt}
\resizebox{\ifdim\width>\textwidth\textwidth\else\width\fi}{!}{%
\begin{tabular}{@{}l l l l l r r r l l r r r r@{}}
\toprule
 &  &  &  &  & \multicolumn{3}{c}{\# HP} &  &  &  &  &  &  \\
\cmidrule(lr){6-8}
\begin{tabular}[c]{@{}l@{}}Model \\ Family\end{tabular} & Comparison & Arms A & Arms B & \begin{tabular}[c]{@{}c@{}}\# Arms \\ A vs B\end{tabular} & Teacher & Student & \begin{tabular}[c]{@{}r@{}}Total \\ A vs B\end{tabular} & Animal & \begin{tabular}[c]{@{}l@{}}Treatment \\ Effect Mode\end{tabular} & \begin{tabular}[c]{@{}r@{}}Diff. in \\ Means\end{tabular} & \begin{tabular}[c]{@{}r@{}}Mean \\ P-value\end{tabular} & \begin{tabular}[c]{@{}r@{}}Win \\ Prob.\end{tabular} & \begin{tabular}[c]{@{}r@{}}Win \\ P-value\end{tabular} \\
\midrule
Pythia & Same-model vs & \{160M, fs1--3, ds1--3, & \{160M, ds1--3\}$^{2}$ & 8 vs 12 & 10 & 30 & 2400 & dog & $\Delta$ & 0.379 & \textbf{1.6

## Which direction transfers more? Arm families that share no cells

Third table: Pythia → OLMo against OLMo → Pythia, 160M → 410M against 410M → 160M, and the
same against the different data order across widths. The two sides of these pairings are read
on different receivers and designs, so no cell-level stratum exists; instead each arm is
summarised against **its own** matched controls and the ARM is the unit
(`transfer_headline.arm_family_test`).

- **Treatment Effect Mode** — `Δ` = the arm's mean gap (treatment − cell control, p(student) −
  p(control)) over its runs; `Δ / prior` = the same gap over its receiver's prior. **Diff. in Means** = mean over A's arms minus mean over B's arms.
- **P-value** — the exact arm-relabelling permutation: every way of splitting the pooled arms
  into sets of A's and B's sizes, two-sided (tail doubling). Bold green = p < 0.05.
- **Paired P-value** — only where the arms pair by model identity (Pythia → OLMo vs its exact
  reverse, model pair by model pair): the exact sign-flip test on the 9 paired differences.
  `–` = the two sides do not pair.
- **40-labelling rule** — a pairing enters only if its arm-relabelling null has **at least 40
  labellings**: with `n` labellings the smallest two-sided p is `2/n`, so below 40 nothing could
  reach 0.05. Dropped pairings are named under the table with their arm counts.
- **A before B** — orientation fixed before the data (small to large first, same order first);
  the SIGN of the difference says which side is stronger.

The LaTeX is written to `notebooks/latex/transfer_direction_table.tex` (frame pickled beside it).

In [6]:
XARCH = REPO / "run_logs/campaigns/xarch/xarch_significance_arms.json"
PAIRINGS = tc.direction_pairings(XARCH)
DIRECTION_RESULTS = tc.direction_results(XARCH, PAIRINGS)   # a pairing failing the 40-labelling rule is never tested
DIRECTIONS, DROPPED = tc.direction_table(DIRECTION_RESULTS)
display(Markdown("\n".join(f"- {n}" for n in tc.direction_notes(DROPPED, pairings=PAIRINGS))))
tc.style_direction_table(DIRECTIONS)

- a pairing enters only when its exact arm-relabelling null has at least 40 labellings (two-sided floor 2/n ≤ 0.05); with fewer, no outcome can reach p < 0.05
- Paired P-value is the exact sign-flip test over arms paired by model identity; – = the two arm sets do not pair, or the pairs are too few for the same floor (2^k ≥ 40)
- Pythia: every small-to-large (160M-sender) arm is summarised on the teachers all of them share (the top-5 arm's five strongest senders × core30), so those arms are exchangeable; large-to-small arms keep their registered per-base teachers
- the per-data-order direction splits are not tabled: same data order has 4 vs 4 arms (70 labellings); different data order (8 vs 6) is subsumed by the pooled small-to-large vs large-to-small row

In [7]:
DIRECTION_TEX = tc.direction_table_latex(DIRECTIONS)
(REPO / "notebooks/latex/transfer_direction_table.tex").write_text(DIRECTION_TEX)
DIRECTIONS.to_pickle(REPO / "notebooks/latex/transfer_direction_table.pkl")   # the render-only path
print(DIRECTION_TEX)

% requires \usepackage{booktabs,graphicx}
\begin{table}[!hp]
\centering
\scriptsize
\setlength{\tabcolsep}{3pt}
\resizebox{\ifdim\width>\textwidth\textwidth\else\width\fi}{!}{%
\begin{tabular}{@{}l l l l l r r r l r r r@{}}
\toprule
 &  &  &  &  & \multicolumn{3}{c}{\# HP} &  &  &  &  \\
\cmidrule(lr){6-8}
\begin{tabular}[c]{@{}l@{}}Model \\ Family\end{tabular} & Comparison & Arms A & Arms B & \begin{tabular}[c]{@{}c@{}}\# Arms \\ A vs B\end{tabular} & Teacher & Student & \begin{tabular}[c]{@{}r@{}}Total \\ A vs B\end{tabular} & \begin{tabular}[c]{@{}l@{}}Treatment \\ Effect Mode\end{tabular} & \begin{tabular}[c]{@{}r@{}}Diff. in \\ Means\end{tabular} & P-value & \begin{tabular}[c]{@{}r@{}}Paired \\ P-value\end{tabular} \\
\midrule
Pythia/OLMo & Small to large vs & \{160M, fs1,3\} $\to$ & Dolma1.7 seed1--3 $\to$ & 9 vs 9 & 16 / 10 / 8 & 35 / 30 & 3360 & $\Delta$ & 3e-3 & 0.42 & 0.41 \\
Cross Arch. & large to small & Dolma1.7 seed1--3 & \{160M, fs1,3\} &  &  &  & vs 2160 & $\Delta$ / pr